# SEQ-01 RNN From Scratch

- Module: 09 Sequence Models
- Topic: explicit recurrence, BPTT, and vanishing/exploding gradients
- Estimated runtime: 8 to 12 minutes on CPU once `torch` and `matplotlib` are installed
- Expected outputs: a manual BPTT gradient check, timestep-wise gradient norms, and a short training run on a toy memory task


## Recurrence and BPTT in one page

A vanilla RNN reuses the same transition map at every timestep:

$$
a_t = W_{xh} x_t + W_{hh} h_{t-1} + b_h,
\qquad
h_t = \tanh(a_t),
\qquad
o_t = W_{hy} h_t + b_y.
$$

If the loss is applied only at the final step, then the hidden-state error still moves backward through every recurrence:

$$
\delta_t = \frac{\partial \mathcal{L}}{\partial a_t}
= \left(W_{hh}^\top \delta_{t+1}\right) \odot \left(1 - h_t^2\right)
\quad \text{for } t < T.
$$

This is ordinary backpropagation on an unrolled graph with tied parameters. The repeated Jacobian products are exactly what create vanishing and exploding gradients.


In [ ]:
import math
import random

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## A toy long-memory task

Each sequence starts with one informative bit at timestep $1$ and then contains only noise. The label asks whether that first bit was positive. This makes the task ideal for visualizing long-range credit assignment: the model must preserve information from the earliest timestep until the end.


In [ ]:
def make_memory_batch(batch_size: int, seq_len: int, noise_scale: float = 0.20):
    first_token = torch.randint(0, 2, (batch_size, 1), dtype=torch.float32)
    informative = first_token * 2.0 - 1.0
    noise = noise_scale * torch.randn(batch_size, seq_len - 1, 1)
    x = torch.cat([informative.unsqueeze(-1), noise], dim=1)
    y = first_token
    return x.to(device), y.to(device)

x_demo, y_demo = make_memory_batch(batch_size=4, seq_len=10)
print("input shape:", tuple(x_demo.shape))
print("labels:", y_demo.squeeze(-1).tolist())
print("first example:", x_demo[0, :, 0].tolist())


## A minimal RNN classifier

The implementation below does not call `nn.RNN`. It performs the recurrent update explicitly so the hidden-state sequence and the tied parameters remain visible.


In [ ]:
class VanillaRNNClassifier(nn.Module):
    def __init__(self, hidden_size: int = 24):
        super().__init__()
        self.hidden_size = hidden_size
        self.Wxh = nn.Parameter(torch.empty(hidden_size, 1))
        self.Whh = nn.Parameter(torch.empty(hidden_size, hidden_size))
        self.bh = nn.Parameter(torch.zeros(hidden_size))
        self.Why = nn.Parameter(torch.empty(1, hidden_size))
        self.by = nn.Parameter(torch.zeros(1))
        self.reset_parameters()

    def reset_parameters(self, recurrent_scale: float = 0.95):
        nn.init.xavier_uniform_(self.Wxh)
        nn.init.orthogonal_(self.Whh)
        with torch.no_grad():
            self.Whh.mul_(recurrent_scale)
        nn.init.xavier_uniform_(self.Why)
        nn.init.zeros_(self.by)
        nn.init.zeros_(self.bh)

    def forward(self, x: torch.Tensor, return_sequences: bool = False):
        batch_size, seq_len, _ = x.shape
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)
        states = []
        preacts = []

        for t in range(seq_len):
            a = x[:, t] @ self.Wxh.T + h @ self.Whh.T + self.bh
            h = torch.tanh(a)
            states.append(h)
            preacts.append(a)

        logits = h @ self.Why.T + self.by
        if return_sequences:
            return logits.squeeze(-1), states, preacts
        return logits.squeeze(-1)


model = VanillaRNNClassifier(hidden_size=12).to(device)
logits, states, preacts = model(x_demo, return_sequences=True)
print("logits shape:", tuple(logits.shape))
print("number of stored hidden states:", len(states))


## Manual BPTT check against autograd

For a final-step binary loss, the recurrent weight gradient is

$$
\nabla_{W_{hh}} \mathcal{L} = \sum_{t=1}^{T} \delta_t h_{t-1}^\top.
$$

The cell below computes this quantity by hand for one example and compares it with PyTorch autograd.


In [ ]:
single_x, single_y = make_memory_batch(batch_size=1, seq_len=6)
manual_model = VanillaRNNClassifier(hidden_size=5).to(device)
manual_model.reset_parameters(recurrent_scale=0.9)
manual_model.zero_grad(set_to_none=True)

logit, states, preacts = manual_model(single_x, return_sequences=True)
loss = F.binary_cross_entropy_with_logits(logit, single_y.squeeze(-1))
loss.backward()
autograd_grad = manual_model.Whh.grad.detach().clone()

with torch.no_grad():
    h_seq = [state.detach().squeeze(0) for state in states]
    h_prev_seq = [torch.zeros_like(h_seq[0])] + h_seq[:-1]
    W_hh = manual_model.Whh.detach()
    W_hy = manual_model.Why.detach().squeeze(0)

    y_hat = torch.sigmoid(logit.detach()).squeeze(0)
    dloss_dlogit = y_hat - single_y.squeeze(-1).detach().squeeze(0)
    dloss_dh = W_hy * dloss_dlogit

    delta = [torch.zeros_like(h_seq[0]) for _ in h_seq]
    delta[-1] = dloss_dh * (1.0 - h_seq[-1] ** 2)
    for t in range(len(h_seq) - 2, -1, -1):
        delta[t] = (W_hh.T @ delta[t + 1]) * (1.0 - h_seq[t] ** 2)

    manual_grad = torch.zeros_like(W_hh)
    for t, d_t in enumerate(delta):
        manual_grad += torch.outer(d_t, h_prev_seq[t])

max_diff = (manual_grad - autograd_grad).abs().max().item()
print(f"max |manual - autograd| = {max_diff:.6e}")
print("manual gradient sample:")
print(manual_grad[:2, :2])


## Empirical vanishing and exploding gradients

We now keep the task fixed and only vary the recurrent-matrix scale. For each model we retain the gradient of every hidden state and inspect the mean norm of $\partial \mathcal{L} / \partial h_t$.

- `scale < 1` tends to shrink gradients as we move backward.
- `scale > 1` can amplify them sharply.
- `scale \approx 1` is the narrow region where training is least unstable.


In [ ]:
def gradient_through_time(scale: float, seq_len: int = 35, batch_size: int = 96):
    model = VanillaRNNClassifier(hidden_size=24).to(device)
    model.reset_parameters(recurrent_scale=scale)
    x, y = make_memory_batch(batch_size=batch_size, seq_len=seq_len)
    logits, states, _ = model(x, return_sequences=True)
    for h in states:
        h.retain_grad()
    loss = F.binary_cross_entropy_with_logits(logits, y.squeeze(-1))
    loss.backward()
    norms = [h.grad.norm(dim=1).mean().item() for h in states]
    return norms, float(loss.item())

scales = [0.55, 0.95, 1.35]
results = {scale: gradient_through_time(scale) for scale in scales}

plt.figure(figsize=(8, 4))
for scale, (norms, _) in results.items():
    plt.plot(range(1, len(norms) + 1), norms, label=f"recurrent scale={scale}")
plt.yscale("log")
plt.xlabel("timestep")
plt.ylabel("mean ||dL/dh_t||")
plt.title("Gradient norms through time")
plt.legend()
plt.tight_layout()
plt.show()

for scale, (norms, loss_value) in results.items():
    print(f"scale={scale:.2f} | first-step grad={norms[0]:.3e} | final-step grad={norms[-1]:.3e} | loss={loss_value:.4f}")


## Training on the memory task

A small vanilla RNN can solve a short-horizon version of the task, but it benefits from careful initialization and gradient clipping. Clipping is a stabilization tool, not a cure for the architecture.


In [ ]:
train_model = VanillaRNNClassifier(hidden_size=32).to(device)
train_model.reset_parameters(recurrent_scale=0.95)
optimizer = torch.optim.Adam(train_model.parameters(), lr=3e-3)

history = []
for step in range(1, 181):
    x_batch, y_batch = make_memory_batch(batch_size=64, seq_len=20)
    logits = train_model(x_batch)
    loss = F.binary_cross_entropy_with_logits(logits, y_batch.squeeze(-1))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(train_model.parameters(), max_norm=1.0)
    optimizer.step()

    with torch.no_grad():
        predictions = (torch.sigmoid(logits) > 0.5).float()
        accuracy = (predictions == y_batch.squeeze(-1)).float().mean().item()
    history.append((step, float(loss.item()), accuracy))

    if step % 45 == 0:
        print(f"step {step:3d} | loss={loss.item():.4f} | accuracy={accuracy:.3f}")

steps, losses, accuracies = zip(*history)
plt.figure(figsize=(8, 4))
plt.plot(steps, losses, label="binary cross-entropy")
plt.plot(steps, accuracies, label="accuracy")
plt.xlabel("training step")
plt.title("Short-horizon training on the memory task")
plt.legend()
plt.tight_layout()
plt.show()


## Recap and extension prompts

What this notebook established:

- the recurrence is explicit and parameter sharing is visible;
- BPTT matches ordinary backpropagation on an unrolled graph;
- repeated Jacobian products create vanishing and exploding gradients;
- clipping can stabilize optimization but does not remove the long-memory bottleneck.

Reasonable next steps:

1. Replace `tanh` with `ReLU` and inspect how the gradient plot changes.
2. Add losses at every timestep instead of only the final one and compare BPTT behavior.
3. Re-run the memory task with an LSTM and compare the degradation at longer sequence lengths.
